# 23. Merge k Sorted Lists
**Difficulty:** 🔴 Hard · **Topic:** Linked List / Heap · **LeetCode:** https://leetcode.com/problems/merge-k-sorted-lists/

## 💡 Concepts

**Core concept(s):** Repeatedly take the smallest head across all lists — fast with a **min-heap**.

**Why it applies here:** With k sorted lists, the next node in the merged list is the smallest of the k current heads. A heap always surfaces that minimum in O(log k), so each of the N nodes costs O(log k).

**Key intuition:** Keep the k front nodes in a heap; pop the smallest, push its successor.

---

### 📚 What is a Linked List?
A **linked list** is a chain of nodes; each node holds a value and a pointer to the **next** node. Unlike an array there is no index — you can only walk forward from the head.
- **In Python:** a small `ListNode` class with `.val` and `.next`.

### 📚 What is a Heap (Priority Queue)?
A **min-heap** always hands you its smallest item in **O(log n)**. Perfect for repeatedly taking the current minimum across many sources.
- **In Python:** `heapq` on a list.

### 📚 Pointers & the Dummy Node
Linked-list code moves **pointers** (references to nodes). A **dummy** node placed before the head removes annoying "is this the first node?" special cases — you build off `dummy.next` and return it at the end.

---

**Prerequisite knowledge:**
- Heaps (`heapq`).
- Merging sorted sequences.

## 📝 Problem

Merge `k` sorted lists into one sorted list.

> Two approaches: collect-and-sort `O(N log N)` and heap `O(N log k)`.

In [ ]:
from typing import Optional, List

class ListNode:
    """A node in a singly linked list: a value plus a link to the next node."""
    def __init__(self, val=0, next=None):
        self.val = val                     # the value stored at this node
        self.next = next                   # link to the next node (None at the end)

def build_list(vals):
    """Turn a Python list into a linked list; return its head."""
    dummy = ListNode(); cur = dummy        # dummy node avoids special-casing the first node
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

def to_list(head):
    """Turn a linked list back into a Python list (handy for printing / assertions)."""
    out = []
    while head:
        out.append(head.val); head = head.next
    return out

### Approach 1 — Collect & Sort (worst)

**Idea:** Dump all values into an array, sort, rebuild a list.

**Time:** `O(N log N)`. **Space:** `O(N)`.

In [ ]:
def merge_k_brute(lists):
    vals = []
    for node in lists:                     # collect every value from every list
        while node:
            vals.append(node.val); node = node.next
    vals.sort()                            # sort them all together
    dummy = ListNode(); cur = dummy        # rebuild one sorted linked list
    for v in vals:
        cur.next = ListNode(v); cur = cur.next
    return dummy.next

### Approach 2 — Min-Heap (optimal)

**Idea:** Push each list's head into a heap. Pop the smallest, append it, and push its next node. The index breaks ties so nodes never get compared.

**Time:** `O(N log k)`. **Space:** `O(k)`.

In [ ]:
import heapq

def merge_k_heap(lists):
    heap = []                              # min-heap of the current front node of each list
    for i, node in enumerate(lists):
        if node:
            heapq.heappush(heap, (node.val, i, node))  # (value, list-index tiebreak, node)
    dummy = ListNode(); cur = dummy
    while heap:
        val, i, node = heapq.heappop(heap) # smallest available value across all lists
        cur.next = node; cur = node        # attach it to the merged list
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))  # push that list's next node
    return dummy.next

In [ ]:
# Correctness check
lists = [build_list([1,4,5]), build_list([1,3,4]), build_list([2,6])]
lists2 = [build_list([1,4,5]), build_list([1,3,4]), build_list([2,6])]
assert to_list(merge_k_brute(lists)) == [1,1,2,3,4,4,5,6]
assert to_list(merge_k_heap(lists2)) == [1,1,2,3,4,4,5,6]
assert to_list(merge_k_heap([])) == []
print("All tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio when `n` → `2n` |
|---|---|
| `O(n)`       | ≈ **2×** |
| `O(n log n)` | ≈ **2×** (slightly more) |
| `O(n²)`     | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    k = 10
    lists = [build_list(list(range(i, n, k))) for i in range(k)]
    return (lists,)
solutions = {
    "collect+sort O(N log N)": merge_k_brute,
    "heap        O(N log k)": merge_k_heap,
}
sizes = [2000, 4000, 8000, 16000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Heap to merge many sorted sources:** always pull the global minimum in O(log k).
- **Tie-break tuple:** add an index so the heap never compares the payload objects.
- **Signal:** "merge k sorted ...", "k-way merge", "smallest across k streams".
- **Related problems:** Merge Two Sorted Lists, Kth Smallest in Sorted Matrix, Smallest Range Covering k Lists.
- **Common pitfalls:** (1) heap comparing nodes (add a tiebreaker); (2) forgetting to push the successor.